# Magic Methods in Python

Goal: understand what magic (dunder) methods are as a general concept, then cover the ones not already seen — `__call__`, `__eq__`/`__hash__`, context managers (`__enter__`/`__exit__`), and iterators (`__iter__`/`__next__`).

## 1. What Are Magic Methods, as a General Concept?

**Why we need it:** you've already used several of these without this notebook — `__init__` runs automatically when an object is created, `__str__`/`__repr__` control printing, `__add__` powers `+`, `__len__` powers `len()`. Each time, *you* never called `obj.__init__()` or `obj.__len__()` directly — Python called it for you, triggered by ordinary syntax (`ClassName(...)`, `print(obj)`, `len(obj)`). That's the whole pattern, and it's worth naming explicitly now that you've seen enough examples to generalize it.

**Use case:** any time you want a custom class to plug into a piece of built-in Python syntax — creation (`()`), printing, arithmetic, `len()`, comparisons, calling it like a function, using it in a `with` block, looping over it with `for` — instead of inventing your own differently-named method for each one.

**How to recognize the pattern:** these methods are always named with double underscores on both sides (`__init__`, `__add__`, `__len__` — hence "dunder," short for "double underscore"), and they're never meant to be called directly by your own code. You write them; Python calls them, at the exact moment the matching syntax is used. This notebook covers the ones not already introduced elsewhere: `__call__`, the `__eq__`/`__hash__` pairing, context managers (`__enter__`/`__exit__`), and iterators (`__iter__`/`__next__`).

## 2. `__call__` — Making an Object Callable

**Why we need it:** a function and an object are usually different things — you `call` a function with `f(x)`, and you access an object's data or call its named methods with `obj.method(x)`. But sometimes an object *is* fundamentally "a thing you call," like a configurable counter, a cache, or a small piece of reusable logic that also needs to remember state between calls — something a plain function can't do on its own without extra machinery (like `global` or a closure).

**Use case:** a `Multiplier(3)` object that you then use as `multiplier(10)` to get `30` — it behaves like a function, but it's actually an object that remembers its own configuration (`3`) between calls. This pattern shows up often for decorators, callbacks, and stateful "function-like" utilities.

**How to implement:** define `__call__(self, ...)` on the class. Once it's there, `obj(...)` becomes valid syntax, and Python translates it into `obj.__call__(...)` — the exact same pattern as every other magic method so far, just triggered by parentheses after an object instead of after a class name.

In [1]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, value):
        return value * self.factor


times_three = Multiplier(3)
print(times_three(10))      # calls times_three.__call__(10)
print(times_three(7))

print(callable(times_three))   # confirms Python considers this object "callable"

# A common real use: passing it around anywhere a function would be expected
numbers = [1, 2, 3, 4]
print(list(map(times_three, numbers)))   # map() just needs something callable

30
21
True
[3, 6, 9, 12]


## 3. `__eq__` and `__hash__` — a Gotcha Worth Knowing

**Why we need it:** the operator-overloading notebook showed defining `__eq__` so two objects compare equal by value instead of identity. What it didn't show is a side effect: as soon as you define `__eq__` yourself, Python quietly *removes* the default `__hash__` your class used to have, making instances unhashable — and unhashable objects can't go in a `set` or be used as `dict` keys. This surprises almost everyone the first time it happens.

**Use case:** any class you want to both compare by value (`__eq__`) *and* store in a `set` or use as a `dict` key — e.g. deduplicating a list of `Point` objects by putting them in a `set`.

**How to implement:** if you define `__eq__`, and you also need the object to be hashable, define `__hash__` too — usually by hashing the same fields your `__eq__` compares, wrapped in a `tuple`: `return hash((self.x, self.y))`. This matters because of a rule Python relies on: two objects that are `==` **must** produce the same hash, or dictionaries and sets silently misbehave (e.g. failing to recognize a value you just checked). If instances should never be hashable at all (because they're meant to be mutable, and mutable objects' hashes could change while they sit inside a set), that's a valid choice too — you'd just leave `__hash__` unset in that case rather than accidentally get a broken one.

In [2]:
class BrokenPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y
    # no __hash__ defined -- Python removed the default one because __eq__ is defined


try:
    {BrokenPoint(1, 2)}          # trying to put it in a set
except TypeError as e:
    print(f"TypeError: {e}")


class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point({self.x}, {self.y})"

    def __eq__(self, other):
        return isinstance(other, Point) and self.x == other.x and self.y == other.y

    def __hash__(self):
        return hash((self.x, self.y))   # same fields used in __eq__


points = [Point(1, 1), Point(2, 2), Point(1, 1)]   # (1, 1) appears twice
unique_points = set(points)
print(unique_points)          # duplicate correctly collapsed to one

lookup = {Point(0, 0): "origin"}
print(lookup[Point(0, 0)])    # a DIFFERENT Point object, but same hash+eq -> found

TypeError: unhashable type: 'BrokenPoint'
{Point(1, 1), Point(2, 2)}
origin


## 4. Context Managers: `__enter__` and `__exit__`

**Why we need it:** you've likely used `with open("file.txt") as f: ...` and noticed the file gets closed automatically, even if the code inside raises an exception — you never had to write a `finally: f.close()` yourself. That guarantee doesn't come for free; it comes from `open()` returning an object that implements a specific pair of magic methods. Anything that needs guaranteed setup-then-cleanup (opening/closing, locking/unlocking, connecting/disconnecting) can get the same guarantee.

**Use case:** a database connection that must always be closed, a lock that must always be released, a timer that should measure the duration of a block of code — anything where "make sure the cleanup step runs, no matter what happens inside" matters.

**How to implement:** define `__enter__(self)`, which runs when the `with` block starts and returns whatever should be bound to the `as` variable, and `__exit__(self, exc_type, exc_value, traceback)`, which runs when the block ends — normally *or* because of an exception. Python calls `__exit__` even if the block raised an error, which is exactly the guarantee `finally` gives you, just packaged onto the object itself instead of repeated at every call site.

In [3]:
class ManagedResource:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print(f"Opening {self.name}")
        return self          # this becomes the value bound to `as ...`

    def __exit__(self, exc_type, exc_value, traceback):
        print(f"Closing {self.name}")
        if exc_type is not None:
            print(f"  (closed after an exception: {exc_type.__name__}: {exc_value})")
        return False          # False means "don't suppress the exception" -- let it propagate


print("--- normal case ---")
with ManagedResource("database connection") as res:
    print(f"Using {res.name}")

print("\n--- exception case ---")
try:
    with ManagedResource("file handle") as res:
        print(f"Using {res.name}")
        raise ValueError("something went wrong inside the block")
except ValueError as e:
    print(f"Caught outside the with-block: {e}")
    # notice "Closing file handle" was still printed above, before this line ran

--- normal case ---
Opening database connection
Using database connection
Closing database connection

--- exception case ---
Opening file handle
Using file handle
Closing file handle
  (closed after an exception: ValueError: something went wrong inside the block)
Caught outside the with-block: something went wrong inside the block


## 5. Iterators: `__iter__` and `__next__`

**Why we need it:** `for x in something:` already works on lists, strings, and dicts, and the operator-overloading notebook showed that defining `__getitem__` alone is enough to make a custom class loop-able too. But `__getitem__` assumes your data is indexable by position (`0, 1, 2, ...`) and fully exists ahead of time. Sometimes what you're iterating over is *generated on demand* — the next value doesn't exist until you ask for it, and there may not be a meaningful "index" at all (e.g. counting upward forever, or reading lines from a live stream).

**Use case:** a `Countdown` that produces `5, 4, 3, 2, 1` one at a time without pre-building a list `[5, 4, 3, 2, 1]` first — useful when the sequence is large, infinite, or expensive to generate all at once.

**How to implement:** define `__iter__(self)`, which Python calls once at the start of a `for` loop and which should return an object with a `__next__` method (often just `self`), and `__next__(self)`, which Python calls once per loop iteration to produce the next value. When there are no more values, `__next__` must raise `StopIteration` — that specific exception is exactly how a `for` loop knows to stop; it's not an error, it's the iteration protocol's normal "done" signal.

In [4]:
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self          # this object knows how to produce its own next value

    def __next__(self):
        if self.current <= 0:
            raise StopIteration      # signals "no more values" -- the for-loop stops cleanly
        value = self.current
        self.current -= 1
        return value


for n in Countdown(5):
    print(n)

# a for-loop is really just repeatedly calling __next__() until StopIteration
c = Countdown(3)
it = iter(c)          # calls c.__iter__()
print(next(it))       # calls it.__next__()
print(next(it))
print(next(it))
try:
    print(next(it))    # nothing left -- raises StopIteration
except StopIteration:
    print("StopIteration: no more values")

5
4
3
2
1
3
2
1
StopIteration: no more values


## 6. Putting It Together: a `Timer` Context Manager

**Why we need it:** to see a realistic case where a class exists *specifically* to plug into `with`, combining `__enter__`/`__exit__` with `__repr__` for a clean summary — a small utility you'd genuinely reach for while developing.

**Use case:** measuring how long a block of code takes to run, without manually writing `start = time.time()` ... `end = time.time()` ... `print(end - start)` around every piece of code you want to time, and without forgetting to record the end time if the block raises an exception partway through.

**How it's implemented here:** `__enter__` records the start time and returns `self` (so `as t` gives you the `Timer` object itself). `__exit__` records the end time — guaranteed to run whether the block succeeded or raised — and always returns `False`, since a timing utility has no business swallowing errors from the code it's timing. `__repr__` gives a readable summary once timing is done.

In [5]:
import time


class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.end = time.perf_counter()
        self.elapsed = self.end - self.start
        return False   # never suppress exceptions from the timed code

    def __repr__(self):
        return f"Timer(elapsed={self.elapsed:.4f}s)"


with Timer() as t:
    total = sum(i * i for i in range(1_000_000))   # something that takes a small but real amount of time

print(f"result: {total}")
print(t)

try:
    with Timer() as t2:
        raise RuntimeError("boom")
except RuntimeError as e:
    print(f"caught: {e}")
print(t2)   # __exit__ still ran, so elapsed time was recorded even though the block failed

result: 333332833333500000
Timer(elapsed=0.0377s)
caught: boom
Timer(elapsed=0.0000s)
